## Data-driven EIT Reconstruction — Ridge Regression

Learns the Dräger PulmoVista 500 reconstruction mapping directly from
paired `.eit` / `.bin` recordings.

- **v1**: input = `vv` (208 calibrated transimpedances)
- **v1b**: input = `[trans_A, trans_B]` (416 raw ADC features)

Train on patient01, test on patient02 (cross-patient generalisation).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from fasteit.reconstruction.data_prep import load_paired, normalize
from fasteit.reconstruction.metrics import (
    correlation_per_frame,
    mse_per_frame,
    summary_metrics,
)
from fasteit.reconstruction.ridge_model import RidgeReconstructor

## 1. Load paired data

In [ ]:
DATA = Path("../src/fasteit/test_files")

# Patient 01 — training
X_train_raw, Y_train_raw = load_paired(
    DATA / "patient01.eit", DATA / "patient01.bin", input_mode="vv"
)
print(f"Patient01: X={X_train_raw.shape}, Y={Y_train_raw.shape}")

# Patient 02 — test
X_test_raw, Y_test_raw = load_paired(
    DATA / "patient02.eit", DATA / "patient02.bin", input_mode="vv"
)
print(f"Patient02: X={X_test_raw.shape}, Y={Y_test_raw.shape}")

## 2. Per-file normalisation

Subtract the mean of the first 50 frames (stable baseline) from each
recording independently. This removes absolute impedance differences
between patients and makes the data safely combinable.

In [ ]:
N_REF = 50

X_train, ref_x_train = normalize(X_train_raw, n_ref=N_REF)
Y_train, ref_y_train = normalize(Y_train_raw, n_ref=N_REF)

X_test, ref_x_test = normalize(X_test_raw, n_ref=N_REF)
Y_test, ref_y_test = normalize(Y_test_raw, n_ref=N_REF)

print(f"Train: X mean={X_train[:50].mean():.2e} (should be ~0)")
print(f"Test:  X mean={X_test[:50].mean():.2e} (should be ~0)")
print(f"Train range: X [{X_train.min():.4f}, {X_train.max():.4f}]")
print(f"Test  range: X [{X_test.min():.4f}, {X_test.max():.4f}]")

## 3. Train Ridge v1 (vv → pixels)

In [ ]:
model_v1 = RidgeReconstructor(alpha=1.0).fit(X_train, Y_train)

r2_train = model_v1.score(X_train, Y_train)
r2_test = model_v1.score(X_test, Y_test)

print(f"v1 (vv, 208 features)")
print(f"  R² train: {r2_train:.4f}")
print(f"  R² test:  {r2_test:.4f}")
print(f"  Weights:  {model_v1.coef_.shape}")

## 4. Train Ridge v1b (trans_A + trans_B → pixels)

In [ ]:
# Reload with raw features
X_train_raw_416, _ = load_paired(
    DATA / "patient01.eit", DATA / "patient01.bin", input_mode="raw"
)
X_test_raw_416, _ = load_paired(
    DATA / "patient02.eit", DATA / "patient02.bin", input_mode="raw"
)

X_train_416, _ = normalize(X_train_raw_416, n_ref=N_REF)
X_test_416, _ = normalize(X_test_raw_416, n_ref=N_REF)

model_v1b = RidgeReconstructor(alpha=1.0).fit(X_train_416, Y_train)

r2_train_b = model_v1b.score(X_train_416, Y_train)
r2_test_b = model_v1b.score(X_test_416, Y_test)

print(f"v1b (trans_A + trans_B, 416 features)")
print(f"  R² train: {r2_train_b:.4f}")
print(f"  R² test:  {r2_test_b:.4f}")
print(f"  Weights:  {model_v1b.coef_.shape}")

## 5. Comparison table

In [ ]:
Y_pred_v1 = model_v1.predict(X_test)
Y_pred_v1b = model_v1b.predict(X_test_416)

m_v1 = summary_metrics(Y_test, Y_pred_v1)
m_v1b = summary_metrics(Y_test, Y_pred_v1b)

print(f"{'Metric':<25s} {'v1 (208)':>10s} {'v1b (416)':>10s}")
print("-" * 47)
for key in m_v1:
    print(f"{key:<25s} {m_v1[key]:>10.4f} {m_v1b[key]:>10.4f}")

## 6. Global signal overlay

The global EIT signal (sum of all pixels per frame) is the most
clinically relevant waveform. If the model works, the predicted
curve should closely follow the .bin ground truth.

In [ ]:
g_true = Y_test.sum(axis=1)
g_v1 = Y_pred_v1.sum(axis=1)
g_v1b = Y_pred_v1b.sum(axis=1)

# Show first 500 frames (~10 s)
fs = 50.0
n_show = 500
t = np.arange(n_show) / fs

fig, axes = plt.subplots(2, 1, figsize=(14, 5), layout="constrained", sharex=True)

axes[0].plot(t, g_true[:n_show], lw=0.9, label=".bin (ground truth)", color="C1")
axes[0].plot(t, g_v1[:n_show], lw=0.9, label="v1 (vv, 208)", color="C0", alpha=0.8)
axes[0].set_ylabel("Global signal (Δpixels)")
axes[0].set_title("Ridge v1 — global signal overlay (patient02, test set)")
axes[0].legend()

axes[1].plot(t, g_true[:n_show], lw=0.9, label=".bin (ground truth)", color="C1")
axes[1].plot(t, g_v1b[:n_show], lw=0.9, label="v1b (raw, 416)", color="C2", alpha=0.8)
axes[1].set_xlabel("Time [s]")
axes[1].set_ylabel("Global signal (Δpixels)")
axes[1].set_title("Ridge v1b — global signal overlay (patient02, test set)")
axes[1].legend()

plt.show()

## 7. Image comparison — single frame

Side-by-side: .bin ground truth vs v1 prediction vs v1b prediction
at an inspiratory peak (max global signal in the first 200 frames).

In [ ]:
# Find an inspiratory peak in test data
peak_idx = int(np.argmax(g_true[:200]))

img_true = Y_test[peak_idx].reshape(32, 32)
img_v1 = Y_pred_v1[peak_idx].reshape(32, 32)
img_v1b = Y_pred_v1b[peak_idx].reshape(32, 32)

vmax = np.max(np.abs(img_true))

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), layout="constrained")

for ax, img, title in zip(
    axes,
    [img_true, img_v1, img_v1b],
    [".bin ground truth", "v1 (vv, 208)", "v1b (raw, 416)"],
    strict=True,
):
    im = ax.imshow(img, cmap="RdBu_r", vmin=-vmax, vmax=vmax, origin="lower")
    ax.set_title(title, fontsize=10)
    ax.axis("off")

fig.colorbar(im, ax=axes, shrink=0.8, label="Δ pixel")
fig.suptitle(f"Inspiratory peak — frame {peak_idx} (t={peak_idx/fs:.2f} s)")
plt.show()

## 8. Spatial correlation distribution

Histogram of per-frame Pearson correlation between predicted and
ground-truth images. A tight distribution near 1.0 means the model
consistently reproduces the spatial pattern across all frames.

In [ ]:
corr_v1 = correlation_per_frame(Y_test, Y_pred_v1)
corr_v1b = correlation_per_frame(Y_test, Y_pred_v1b)

fig, ax = plt.subplots(figsize=(8, 3), layout="constrained")
ax.hist(corr_v1, bins=80, alpha=0.6, label=f"v1 (mean={corr_v1.mean():.3f})", color="C0")
ax.hist(corr_v1b, bins=80, alpha=0.6, label=f"v1b (mean={corr_v1b.mean():.3f})", color="C2")
ax.set_xlabel("Per-frame spatial correlation")
ax.set_ylabel("Count")
ax.set_title("Spatial fidelity — per-frame Pearson r (test set)")
ax.legend()
plt.show()

## 9. MSE over time

MSE per frame plotted over time. Spikes indicate frames where the
model struggles — typically during rapid transitions (e.g. cough,
suction, position change).

In [ ]:
mse_v1 = mse_per_frame(Y_test, Y_pred_v1)
mse_v1b = mse_per_frame(Y_test, Y_pred_v1b)
t_all = np.arange(len(mse_v1)) / fs

fig, ax = plt.subplots(figsize=(14, 3), layout="constrained")
ax.plot(t_all, mse_v1, lw=0.5, label="v1", color="C0", alpha=0.7)
ax.plot(t_all, mse_v1b, lw=0.5, label="v1b", color="C2", alpha=0.7)
ax.set_xlabel("Time [s]")
ax.set_ylabel("MSE per frame")
ax.set_title("Reconstruction error over time (test set)")
ax.legend()
plt.show()

## 10. Error map — mean absolute error per pixel

Shows where the model makes the largest errors spatially.
Expect higher error near the heart region (cardiac artifact)
and at the edges of the lung fields.

In [ ]:
mae_map_v1 = np.abs(Y_test - Y_pred_v1).mean(axis=0).reshape(32, 32)
mae_map_v1b = np.abs(Y_test - Y_pred_v1b).mean(axis=0).reshape(32, 32)

vmax_err = max(mae_map_v1.max(), mae_map_v1b.max())

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5), layout="constrained")

im0 = axes[0].imshow(mae_map_v1, cmap="hot", vmin=0, vmax=vmax_err, origin="lower")
axes[0].set_title("v1 — mean absolute error per pixel")
axes[0].axis("off")

im1 = axes[1].imshow(mae_map_v1b, cmap="hot", vmin=0, vmax=vmax_err, origin="lower")
axes[1].set_title("v1b — mean absolute error per pixel")
axes[1].axis("off")

fig.colorbar(im1, ax=axes, shrink=0.8, label="MAE")
plt.show()

## Summary

| Model | Input | Features | R² test | Global corr | Spatial corr |
|-------|-------|----------|---------|-------------|-------------|
| v1 | `vv` (EIDORS calibrated) | 208 | ? | ? | ? |
| v1b | `[trans_A, trans_B]` (raw) | 416 | ? | ? | ? |

Fill in after running. If both R² > 0.90 and global corr > 0.95,
Ridge regression is sufficient and no neural network is needed.